## Exercise 2 (1 point)
Implement read_remote_csv and read_remote_parquet tools that read a CSV/Parquet file from URL and return it to the LLM as text. Use Polars for this.

You can use any publicly hosted files, or make your own. Public examples:

CSV - our own ApisTox dataset)
Parquet - New York yellow taxi rides (select a few dozen rows for testing)
Test your LLM with the tool, asking a few questions about the dataset. First, you need to provide it with the URL to point to the file.

In [1]:
import io
import json
import os
from typing import Callable

import polars as pl
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

API_KEY = os.getenv("GEMINI_API_KEY")

MODEL = "gemini-2.5-flash-lite"  # 20 req/day free tier
# MODEL = "gemini-3.1-flash-lite-preview"  # test phase, lots of tokens


client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)


In [2]:
def make_llm_request(prompt: str) -> str:
    client = OpenAI(
        api_key=API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    messages = [
        {"role": "developer", "content": "You are a weather assistant."},
        {"role": "user", "content": prompt},
    ]

    tool_definitions, tool_name_to_func = get_tool_definitions()

    # guard: loop limit, we break as soon as we get an answer
    for _ in range(10):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tool_definitions,  # always pass all tools in this example
            tool_choice="auto",
            max_completion_tokens=1000,
            # extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        resp_message = response.choices[0].message
        messages.append(
            {k: v for k, v in resp_message.model_dump().items() if v is not None}
        )

        print(f"Generated message: {resp_message.model_dump()}")
        print()

        # parse possible tool calls (assume only "function" tools)
        if resp_message.tool_calls:
            for tool_call in resp_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # call tool, serialize result, append to messages
                func = tool_name_to_func[func_name]
                func_result = func(**func_args)

                messages.append(
                    {
                        "role": "tool",
                        "content": json.dumps(func_result),
                        "tool_call_id": tool_call.id,
                    }
                )
        else:
            # no tool calls, we're done
            return resp_message.content

    # we should not get here
    last_response = resp_message.content
    return f"Could not resolve request, last response: {last_response}"


def get_tool_definitions() -> tuple[list[dict], dict[str, Callable]]:
    tool_definitions = [
        {
            "type": "function",
            "function": {
                "name": "read_remote_csv",
                "description": "Download a CSV file from a URL and return its contents as text.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "The URL of the CSV file to read.",
                        },
                    },
                    "required": ["url"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "read_remote_parquet",
                "description": "Download a Parquet file from a URL and return its contents as text.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "The URL of the Parquet file to read.",
                        },
                    },
                    "required": ["url"],
                },
            },
        },
    ]

    tool_name_to_callable = {
        "read_remote_csv": read_remote_csv_tool,
        "read_remote_parquet": read_remote_parquet_tool,
    }

    return tool_definitions, tool_name_to_callable


def read_remote_csv_tool(url: str) -> str:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    df = pl.read_csv(io.BytesIO(response.content))
    return df.head(20).write_csv()


def read_remote_parquet_tool(url: str) -> str:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    df = pl.read_parquet(io.BytesIO(response.content))
    return df.head(20).write_csv()


Csv call and request

In [3]:
prompt = "Based on returned data, tell me the domain of this csv dataset (download it first - use tools) , imagine you work with a data geek who is really interested in some unusual observations: https://github.com/j-adamczyk/ApisTox_dataset/blob/master/outputs/dataset_final.csv"
response = make_llm_request(prompt)
print("Prompt:\n", prompt)
print("Response:\n", response)


Generated message: {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'function-call-12971328564702280234', 'function': {'arguments': '{"url":"https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/master/outputs/dataset_final.csv"}', 'name': 'read_remote_csv'}, 'type': 'function'}]}

Generated message: {'content': "The domain of this CSV dataset appears to be related to \nagrochemicals and their toxicity. The columns include information such as chemical names, CAS numbers, toxicity types (e.g., Contact, Oral, Other), and categories like herbicide, fungicide, and insecticide.\n\nThe dataset also contains unusual observations, such as the inclusion of substances like 'Kieselguhr' and 'Calcium carbonate', which might not be typically considered as primary agrochemicals but could be used in formulations or as inert ingredients. Additionally, the dataset lists various forms of salts and oxides of me

Parquet call and request

In [4]:
prompt = "Tell me something interesting about this dataset, imagine you work with a data geek who is really interested in some unusual observations: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
response = make_llm_request(prompt)
print("Prompt:\n", prompt)
print("Response:\n", response)

print()

Generated message: {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'function-call-14844709193787260409', 'function': {'arguments': '{"url":"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"}', 'name': 'read_remote_parquet'}, 'type': 'function'}]}

Generated message: {'content': "The dataset contains 20 entries. The earliest pickup is at 2026-01-01T00:04:59 and the latest dropoff is at 2026-01-01T01:20:19.\n\nHere are a few interesting observations:\n\n*   **Midnight Mania:** There are quite a few trips happening right at the stroke of midnight, with timestamps like 2026-01-01T00:54:04, 2026-01-01T00:34:04, and 2026-01-01T00:57:06. It seems like New Year's Eve celebrations might have extended into the early hours!\n*   **Long Haul:** One trip stands out with a `trip_distance` of 10.2, which is significantly longer than the others. This could be an airport transfer or a